In [ ]:
# 1/5 - Clone TokenVector prototype/cuda-backend branch
import subprocess, os
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print(r.stderr[:1000])
print('HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
import glob
print('cuda_backend files:', glob.glob(d+'/Release/cuda_backend/examples/*'))

In [ ]:
# 2/5 - Run tkv_cuda.py generator on vecadd.tkv
import subprocess, sys, os
root='/content/TokenVectorCuda/Release/cuda_backend'
ex=root+'/examples'
r=subprocess.run([sys.executable, root+'/tkv_cuda.py', ex+'/vecadd.tkv', 'gen', '-o', ex, '--dll', 'tkv_cuda_kernels'], capture_output=True, text=True, timeout=120)
print('gen rc=', r.returncode)
print(r.stdout[:2000]); print(r.stderr[:2000])
cu=ex+'/vecadd.cuda.cu'
print('cu exists=', os.path.exists(cu))
if os.path.exists(cu): print(open(cu).read()[:2000])

In [ ]:
# 3/5 - Build generated .cu with nvcc (verified flags)
import subprocess, shutil, glob, os
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
cu=ex+'/vecadd.cuda.cu'
so=ex+'/tkv_cuda_kernels.so'
nvcc=shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
print('nvcc =', nvcc)
cmd=(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu '
     '-Xcompiler -std=c++20 -Xcompiler -fPIC '
     +cu+' -o '+so+' -lcudart')
p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
print('build rc=', p.returncode)
if p.returncode!=0: print('STDERR:', p.stderr[:4000])
print('so exists=', os.path.exists(so))

In [ ]:
# 4/5 - Load .so and test vec_add / scale kernels via ctypes
import ctypes, numpy as np, time
so='/content/TokenVectorCuda/Release/cuda_backend/examples/tkv_cuda_kernels.so'
lib=ctypes.CDLL(so)
F64=ctypes.c_double; F32=ctypes.c_float; I32=ctypes.c_int; I64=ctypes.c_int64
lib.tkv_launch_vec_add.argtypes=[ctypes.POINTER(F32)]*3+[I32,I32,I32]
lib.tkv_launch_vec_add.restype=I32
lib.tkv_launch_scale.argtypes=[ctypes.POINTER(F32),F32,I32,I32,I32]
lib.tkv_launch_scale.restype=I32
n=1<<20
a=np.random.rand(n).astype(np.float32); b=np.random.rand(n).astype(np.float32); c=np.zeros(n,np.float32)
pa=a.ctypes.data_as(ctypes.POINTER(F32)); pb=b.ctypes.data_as(ctypes.POINTER(F32)); pc=c.ctypes.data_as(ctypes.POINTER(F32))
grid=(n+255)//256; block=256
t0=time.perf_counter(); rc=lib.tkv_launch_vec_add(pa,pb,pc,n,grid,block); dt=(time.perf_counter()-t0)*1000
print('vec_add rc=',rc,' time_ms=',round(dt,3))
err=np.abs(c-(a+b)).max()
print('vec_add max_abs_err=',err,' CORRECT=',err<1e-5)
# scale test
d=a.copy()
t0=time.perf_counter(); rc=lib.tkv_launch_scale(d.ctypes.data_as(ctypes.POINTER(F32)),F32(2.0),n,grid,block); dt=(time.perf_counter()-t0)*1000
err2=np.abs(d-(a*2.0)).max()
print('scale rc=',rc,' time_ms=',round(dt,3),' max_abs_err=',err2,' CORRECT=',err2<1e-5)

In [ ]:
# 5/5 - Summary
print('CUDA BACKEND PROTOTYPE TEST COMPLETE')
print('If vec_add/scale CORRECT=True: kernel gen -> nvcc build -> ctypes launch pipeline works.')
print('Next step: define fused AdamW kernel in .tkv and replace torch_fused.cpp AdamW path.')